In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import fetch_20newsgroups
import numpy as np
from collections import Counter
import os
import tarfile
import urllib.request

# ======================================================
# 1. IMDB DOWNLOAD
# ======================================================
print("Downloading IMDB...")

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
file_path = "imdb.tar.gz"

if not os.path.exists("aclImdb"):
    urllib.request.urlretrieve(url, file_path)
    with tarfile.open(file_path, "r:gz") as tar:
        tar.extractall()

def load_imdb(path, limit=2000):
    texts = []
    labels = []

    for label in ["pos", "neg"]:
        folder = os.path.join(path, "train", label)
        files = os.listdir(folder)[:limit]

        for f in files:
            with open(os.path.join(folder, f), encoding="utf-8") as file:
                texts.append(file.read())
                labels.append(1 if label == "pos" else 0)

    return texts, labels

imdb_texts, imdb_labels = load_imdb("aclImdb")

# ======================================================
# 2. 20 NEWSGROUPS
# ======================================================
news = fetch_20newsgroups(subset="train")

news_texts = news.data[:4000]
news_labels = news.target[:4000]

# ======================================================
# 3. TOKENIZATION
# ======================================================
def tokenize(text):
    return text.lower().split()

# ======================================================
# 4. DATASET CLASS
# ======================================================
class TextDataset(Dataset):

    def __init__(self, texts, labels, vocab, max_len=200):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def encode(self, text):
        return [self.vocab.get(w, 1) for w in tokenize(text)]

    def pad(self, seq):
        if len(seq) < self.max_len:
            return seq + [0] * (self.max_len - len(seq))
        return seq[:self.max_len]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        x = self.pad(self.encode(self.texts[idx]))

        x = torch.tensor(x, dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)

        return x, y

# ======================================================
# 5. MODELE CNN
# ======================================================

# ----------------------------
# 1 WARSTWA
# ----------------------------
class TextCNN1(nn.Module):

    def __init__(self, vocab_size, embed_dim=100, num_classes=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv1 = nn.Conv1d(embed_dim, 128, 3)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))

        x = torch.max(x, dim=2).values

        x = self.dropout(x)

        x = self.fc(x)

        return x

# ----------------------------
# 2 WARSTWY
# ----------------------------
class TextCNN2(nn.Module):

    def __init__(self, vocab_size, embed_dim=100, num_classes=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv1 = nn.Conv1d(embed_dim, 64, 3)
        self.conv2 = nn.Conv1d(64, 128, 3)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))

        x = torch.max(x, dim=2).values

        x = self.dropout(x)

        x = self.fc(x)

        return x

# ----------------------------
# 3 WARSTWY
# ----------------------------
class TextCNN3(nn.Module):

    def __init__(self, vocab_size, embed_dim=100, num_classes=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv1 = nn.Conv1d(embed_dim, 64, 3)
        self.conv2 = nn.Conv1d(64, 128, 3)
        self.conv3 = nn.Conv1d(128, 128, 3)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))

        x = torch.max(x, dim=2).values

        x = self.dropout(x)

        x = self.fc(x)

        return x

# ======================================================
# 6. DEVICE
# ======================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================================
# 7. TESTOWANE MODELE
# ======================================================
models = {
    "1_LAYER": TextCNN1,
    "2_LAYER": TextCNN2,
    "3_LAYER": TextCNN3
}

# ======================================================
# 8. MAIN LOOP
# ======================================================
for dataset_name, (texts, labels) in {
    "IMDB": (imdb_texts, imdb_labels),
    "20NEWS": (news_texts, news_labels)
}.items():

    print("\n====================================")
    print(f"DATASET: {dataset_name}")
    print("====================================")

    # ----------------------------
    # VOCAB
    # ----------------------------
    counter = Counter()

    for t in texts:
        counter.update(tokenize(t))

    vocab = {
        w: i + 2
        for i, (w, _) in enumerate(counter.most_common(15000))
    }

    vocab["<pad>"] = 0
    vocab["<unk>"] = 1

    # ----------------------------
    # SPLIT
    # ----------------------------
    split = int(0.8 * len(texts))

    train_ds = TextDataset(
        texts[:split],
        labels[:split],
        vocab
    )

    test_ds = TextDataset(
        texts[split:],
        labels[split:],
        vocab
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=32,
        shuffle=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=32
    )

    # ==================================================
    # TEST WSZYSTKICH MODELI
    # ==================================================
    for model_name, ModelClass in models.items():

        print("\n------------------------------------")
        print(f"MODEL: {model_name}")
        print("------------------------------------")

        num_classes = len(set(labels))

        model = ModelClass(
            len(vocab),
            num_classes=num_classes
        ).to(device)

        print("\nARCHITEKTURA:")
        print(model)

        loss_fn = nn.CrossEntropyLoss()

        optimizer = optim.Adam(
            model.parameters(),
            lr=0.001
        )

        # ----------------------------
        # EARLY STOPPING
        # ----------------------------
        max_epochs = 50

        best_acc = 0

        patience = 3
        counter_stop = 0

        # ----------------------------
        # TRAINING
        # ----------------------------
        for epoch in range(max_epochs):

            model.train()

            total_loss = 0

            for X, y in train_loader:

                X = X.to(device)
                y = y.to(device)

                optimizer.zero_grad()

                out = model(X)

                loss = loss_fn(out, y)

                loss.backward()

                optimizer.step()

                total_loss += loss.item()

            # ----------------------------
            # EVALUATION
            # ----------------------------
            model.eval()

            correct = 0
            total = 0

            with torch.no_grad():

                for X, y in test_loader:

                    X = X.to(device)
                    y = y.to(device)

                    preds = model(X).argmax(dim=1)

                    correct += (preds == y).sum().item()

                    total += y.size(0)

            acc = correct / total

            print(f"\nEpoch {epoch+1}")
            print(f"Loss: {total_loss:.4f}")
            print(f"Accuracy: {acc:.4f}")

            # ----------------------------
            # SAVE BEST
            # ----------------------------
            if acc > best_acc:

                best_acc = acc

                counter_stop = 0

                torch.save(
                    model.state_dict(),
                    f"best_{dataset_name}_{model_name}.pth"
                )

                print("NOWY NAJLEPSZY MODEL")

            else:

                counter_stop += 1

                print(
                    f"Brak poprawy "
                    f"({counter_stop}/{patience})"
                )

            # ----------------------------
            # EARLY STOPPING
            # ----------------------------
            if counter_stop >= patience:

                print("EARLY STOPPING")

                break

        print(
            f"\nBEST ACCURACY "
            f"({dataset_name}, {model_name}): "
            f"{best_acc:.4f}"
        )


DATASET: IMDB

------------------------------------
MODEL: 1_LAYER
------------------------------------

ARCHITEKTURA:
TextCNN1(
  (embedding): Embedding(15002, 100)
  (conv1): Conv1d(100, 128, kernel_size=(3,), stride=(1,))
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

Epoch 1
Loss: 70.0178
Accuracy: 0.1650
NOWY NAJLEPSZY MODEL

Epoch 2
Loss: 59.2563
Accuracy: 0.4275
NOWY NAJLEPSZY MODEL

Epoch 3
Loss: 51.1558
Accuracy: 0.3738
Brak poprawy (1/3)

Epoch 4
Loss: 43.8704
Accuracy: 0.3113
Brak poprawy (2/3)

Epoch 5
Loss: 41.1303
Accuracy: 0.5550
NOWY NAJLEPSZY MODEL

Epoch 6
Loss: 35.3278
Accuracy: 0.5775
NOWY NAJLEPSZY MODEL

Epoch 7
Loss: 29.3506
Accuracy: 0.6600
NOWY NAJLEPSZY MODEL

Epoch 8
Loss: 27.2289
Accuracy: 0.6675
NOWY NAJLEPSZY MODEL

Epoch 9
Loss: 22.8689
Accuracy: 0.6175
Brak poprawy (1/3)

Epoch 10
Loss: 20.6190
Accuracy: 0.6138
Brak poprawy (2/3)

Epoch 11
Loss: 18.5473
Accuracy: 0.5625
Brak popr